# video-app lab 04 - memory watch + tiny clip
Two different pools get confused: **VRAM** (GPU, 14.6 GB on T4) and **RAM** (system, ~51 GB).
Colab's crash message says *RAM*, so the system pool was the one exhausted - not the GPU.
This notebook prints BOTH before and after every step, so we see exactly which stage eats what.
Tiny settings first: 8 frames, 384x384. Goal = a clip PLUS the numbers.


In [ ]:
!pip -q install "diffusers>=0.33" transformers accelerate safetensors psutil imageio imageio-ffmpeg


In [ ]:
import os, gc, time, torch, psutil
def mem(tag):
    vm = psutil.virtual_memory()
    vram = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    vram_t = torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else 0
    print(f'{tag:26s} RAM {vm.used/1024**3:5.1f}/{vm.total/1024**3:.1f} GB   VRAM {vram:4.2f}/{vram_t:.1f} GB')
mem('start')
print('cuda:', torch.cuda.is_available(), '| free disk GB:', round(__import__('shutil').disk_usage('/').free/1024**3,1))


In [ ]:
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
DTYPE = torch.float16
mem('before adapter')
adapter = MotionAdapter.from_pretrained('guoyww/animatediff-motion-adapter-v1-5-2', torch_dtype=DTYPE)
mem('after adapter')
pipe = AnimateDiffPipeline.from_pretrained('emilianJR/epiCRealism', motion_adapter=adapter, torch_dtype=DTYPE)
mem('after base pipeline')
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')
pipe.load_lora_weights('ByteDance/AnimateDiff-Lightning', weight_name='animatediff_lightning_4step_diffusers.safetensors', adapter_name='lightning')
pipe.set_adapters(['lightning'], adapter_weights=[1.0])
mem('after lightning lora')
pipe.enable_vae_slicing(); pipe.enable_attention_slicing()
pipe.to('cuda')
mem('after to cuda')


In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display
PROMPT = 'cinematic photo of a red fox trotting through fresh snow in a pine forest, morning light, shallow depth of field, steady camera'
mem('before generate')
t0=time.time()
frames = pipe(prompt=PROMPT, negative_prompt='bad quality, blurry, distorted, watermark',
              num_frames=8, height=384, width=384, guidance_scale=1.0, num_inference_steps=4,
              generator=torch.Generator('cpu').manual_seed(42)).frames[0]
print('GENERATED in', round(time.time()-t0), 's')
mem('after generate')
export_to_video(frames, '/content/clip_04_fox.mp4', fps=8)
print('file MB:', round(os.path.getsize('/content/clip_04_fox.mp4')/1024**2,1))
display(Video('/content/clip_04_fox.mp4', embed=True, width=384))
del pipe, adapter; gc.collect(); torch.cuda.empty_cache()
mem('after cleanup')


### How to read it
RAM climbing on `after base pipeline` = model load is the hog. RAM flat, VRAM climbing = GPU fine.
If this passes, we scale frames/resolution up one step at a time, never two at once.
